In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2024-08-07 06:08:23.533549: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-07 06:08:23.637421: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-07 06:08:23.639378: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-07 06:08:25.002962: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


matrix_approx_zeshel.py


# Open Data loader & context

In [2]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [3]:
from sklearn.cluster import KMeans

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, shuffle=False, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        if shuffle:
            np.random.shuffle(self.relevs)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def get_kmeans_games(self, all_from_labels=True):
        X = self.get_relevs("train").T

        k_func = (
            (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
            if not all_from_labels else
            (lambda C : from_labels(X, C.labels_))
        )
        K_KMeans = k_func(
            KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
        )
        
        return K_KMeans
    
    def set_kmeans_games_as_key(self, *args, **kwargs):
        self.key_games = self.get_kmeans_games(*args, **kwargs)
        return self
    

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

# Games Data loader & context

In [4]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            
            self.key_size = key_size
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def get_kmeans_games(self, all_from_labels=True):
            X = self.get_relevs("train").T

            k_func = (
                (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
                if not all_from_labels else
                (lambda C : from_labels(X, C.labels_))
            )
            K_KMeans = k_func(
                KMeans(n_clusters=self.key_size, random_state=0).fit(X)  #, n_init="auto").fit(X)
            )

            return K_KMeans

        def set_kmeans_games_as_key(self, *args, **kwargs):
            self.key_games = self.get_kmeans_games(*args, **kwargs)
            return self

        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Models

In [5]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        self.trus_top = dict()
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):    
        recs = self.recommend(t)
        
        if isinstance(recs, list):
            recs = np.array(recs)
        
        mse = np.mean((recs - self.get_user_scores(t)) ** 2)
        
        if isinstance(recs, tf.Tensor):
            recs = tf.argsort(-recs, axis=1)[:, :topsize].numpy()
        else:
            recs = np.argsort(-recs, axis=1)[:, :topsize]
        
        if (t, topsize) not in self.trus_top:
            trus = self.get_user_scores(t)
            trus = np.argsort(-trus, axis=1)[:, :topsize]
            self.trus_top[(t, topsize)] = trus
            
        trus = self.trus_top[(t, topsize)]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [6]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False,
    "Winit": "glorot0.01",
    "Wfreeze": False,
    "TEinit": "legacy",
    "normalize_embs": True
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in sorted(ctx.key_games)])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["TEinit"] == "legacy":
                value = self.embed_games
            elif self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                value = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print(type(value))
            else:
                assert False, "unknown TEinit: " + str(self.fit_kwargs["TEinit"])
                
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(value, dtype=tf.float32),
                trainable=True
            )
        else:
            if self.fit_kwargs["TEinit"] == "anncur":
                key_cols_idx = np.array(sorted(ctx.key_games))
                self.embed_games = (
                    np.linalg.pinv(ctx.get_relevs("train")[:, key_cols_idx])
                    @ ctx.get_relevs("train")
                ).T
                print("ANNCur init")
            
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        if self.fit_kwargs["normalize_embs"]:
            return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        else:
            return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            if self.fit_kwargs["normalize_embs"]:
                return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
            else:
                return self.embed_games 
        else:
            return self.trainable_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        
        
        if self.p["Winit"] == "glorot0.01":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
            values = values / 100.  
        elif self.p["Winit"] == "glorot":
            initializer = tf.keras.initializers.GlorotUniform()
            values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        elif self.p["Winit"] == "eye":
            values = np.eye(train_user_embs.shape[1], game_embs.shape[1])
        else:
            assert False, "init is not known:" + str(self.p["init"])
            
        self.W = tf.Variable(values, trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ tf.transpose(game_embs_)

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix and (not self.p["Wfreeze"]):
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [7]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [8]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

In [9]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [10]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

# support items choice

In [13]:
# setup:
# 1000 items, 1000 + 1000 queries
recall_k = 5
n_runs = 10
n_items_per_run = 1000
test_item_cnt = 100
item_cnt_range = list(range(5, 101, 5))
assert test_item_cnt in item_cnt_range

In [10]:
from collections import defaultdict
from itertools import combinations

from matplotlib import pyplot as plt
import numpy as np
import os
from sklearn.metrics import top_k_accuracy_score
from sklearn.cluster import KMeans
from scipy.stats import mannwhitneyu, ttest_rel

In [ ]:
#with open("collections_10000_items/test.npy", "rb") as fin:
#    all_test = np.load(fin)
#with open("collections_10000_items/train.npy", "rb") as fin:
#    all_train = np.load(fin)

#assert all_train.shape[0] == n_runs * n_items_per_run
#assert all_test.shape[0] == n_runs * n_items_per_run


In [12]:
def eval_items(items, train, test):
    item_embeds = train.dot(np.linalg.pinv(train[items]))
    approx_test_rel = item_embeds.dot(test[items])
    approx_train_rel = item_embeds.dot(train[items])
    best_items = np.argsort(test, axis=0)[-1]
    recall = top_k_accuracy_score(best_items, approx_test_rel.T, k=recall_k, labels=np.arange(train.shape[0]))
    test_mse = np.mean((test - approx_test_rel) ** 2)
    train_mse = np.mean((train - approx_train_rel) ** 2)
    return recall, test_mse, train_mse

def get_rnd_items(n_items, train):
    return np.random.choice(train.shape[0], n_items, replace=False)

def get_kmeans_items(n_items, train):
    kmeans = KMeans(n_clusters=n_items, n_init="auto").fit(train)
    items = []
    for center in kmeans.cluster_centers_:
        distances = ((train - center) ** 2).sum(axis=1)
        assert distances.shape == (train.shape[0],)
        for item_id in np.argsort(distances):
            if item_id not in items:
                items.append(item_id)
                break
    return np.array(items, dtype=np.int64)

greedy_items = []
def get_greedy(n_items, train, lazy=True):
    global greedy_items
    if not lazy:
        greedy_items = []
    if len(greedy_items) >= n_items:
        return np.array(greedy_items[:n_items], dtype=np.int64)
    
    gram = train.dot(train.T)
    gram /= gram.mean()
    items = []
    remaining_items = np.ones(train.shape[0], dtype="bool")
    for t in range(n_items):
        scores = (gram ** 2).sum(axis=1)
        assert np.allclose(scores[~remaining_items], np.zeros_like(scores[~remaining_items])), (
            t, scores[~remaining_items]
        )
        scores[remaining_items] /= gram[remaining_items, remaining_items]
        if max(scores) < 1e-9:
            break
        new_item = scores.argmax()
        assert remaining_items[new_item]
        coefs = gram[new_item] / np.sqrt(gram[new_item, new_item])
        diff = coefs.reshape(-1, 1).dot(coefs.reshape(1, -1))
        gram -= diff
        assert np.allclose(gram[new_item], np.zeros_like(gram[new_item]), atol=1e-6), (
            t, gram[new_item].std()
        )
        items.append(new_item)
        remaining_items[new_item] = False
    assert len(items) == n_items
    greedy_items.clear()
    greedy_items.extend(items)
    return np.array(items, dtype=np.int64)


### fast-a

In [13]:
def coitem_algorithm(n_support, candidate_items, target_items, min_item_rel_norm=1e-5, eps=1e-6):
    support_items = []
    support_items_scores = []
    n_c, n_q = candidate_items.shape
    n_t = target_items.shape[0]

    candidate_item_squared_norms = (candidate_items ** 2).sum(axis=1)
    min_item_norm = min_item_rel_norm * np.sqrt(candidate_item_squared_norms.mean())
    
    candidate_coitems = candidate_items.dot(
        target_items.T.dot(target_items)
    )
    orthonormed_support_items = np.zeros((n_support, n_q))
    remaining_items = np.ones(candidate_items.shape[0], dtype="bool")
    for t in tqdm.tqdm_notebook(range(n_support)):
        scores = (candidate_coitems * candidate_items).sum(axis=1)
        remaining_items &= (candidate_item_squared_norms >= min_item_norm ** 2)
        scores[remaining_items] /= candidate_item_squared_norms[remaining_items]
        scores[~remaining_items] = 0
        
        new_item_id = np.argmax(scores)
        assert remaining_items[new_item_id]
        support_items.append(new_item_id)
        support_items_scores.append(scores[new_item_id] / (n_t * n_q))
        remaining_items[new_item_id] = False
        
        new_item = candidate_items[new_item_id].copy()
        new_item -= orthonormed_support_items[:t].T.dot(
            orthonormed_support_items[:t].dot(new_item)
        )
        assert np.allclose((new_item ** 2).sum(), candidate_item_squared_norms[new_item_id], atol=eps)
        new_item /= np.sqrt((new_item ** 2).sum())
        orthonormed_support_items[t] = new_item
        new_coitem = candidate_coitems[new_item_id] / np.sqrt(candidate_item_squared_norms[new_item_id])
        
        coefs = (candidate_items * new_item).sum(axis=1)
        candidate_item_squared_norms -= coefs ** 2
        assert np.all(candidate_item_squared_norms[remaining_items] > -eps)
        
        candidate_coitems -= coefs.reshape((-1, 1)).dot(new_coitem.reshape((1, -1)))
        cocoefs = (candidate_coitems * new_item).sum(axis=1, keepdims=True)
        candidate_coitems -= cocoefs.dot(new_item.reshape((1, -1)))
    return np.array(support_items, dtype=np.int64), np.array(support_items_scores)

In [14]:
ctx.key_games

NameError: name 'ctx' is not defined

In [11]:
LS = ["train", "validation", "test"]

RD = {
    t: np.load(f'./qs_ps_embs_{t}.npz')
    for t in LS
}

In [12]:
qs_emb = np.vstack([
    RD[k]["qs_emb"]
    for k in LS
])

qs_emb.shape

(102023, 768)

In [13]:
ps_emb = np.vstack([
    RD[k]["ps_emb"]
    for k in LS
])

ps_emb.shape

(785843, 768)

In [35]:
R = qs_emb[-500:, :] @ ps_emb.T
R.shape

(500, 785843)

In [38]:
ctx = EvalContextRelevs(
    R,
    det_attempts=0
)

Best det =  0.0
Current de =  0.0
100 249 150


In [39]:
# eval_clustering(ctx, all_from_labels=True)

In [40]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

<ipython-input-17-a13f9b2faac9>:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7878714859437751 0.0005458405 249
np.mean(results), mse, len(results) =  0.4672 0.0017797131 150


(0.7878714859437751, 0.4672)

In [44]:
print(ctx.key_games)

[278664, 299295, 209934, 783546, 167140, 18418, 591333, 732591, 548523, 625832, 460686, 405446, 406901, 357981, 605785, 45117, 470756, 103429, 47122, 347758, 516807, 210292, 557032, 305837, 428734, 356506, 574879, 552365, 129265, 482408, 205157, 77431, 453482, 472881, 655623, 640264, 555583, 486454, 168111, 93557, 187927, 53213, 260736, 344702, 313955, 256429, 82184, 192084, 437558, 320044, 341419, 405182, 506419, 251645, 673867, 172637, 537014, 738390, 244348, 233589, 354684, 673224, 144868, 573330, 400094, 754618, 737980, 66137, 365690, 554733, 385256, 172970, 486029, 228743, 561626, 403828, 439128, 686529, 649754, 172412, 82972, 400101, 614759, 388260, 243977, 497371, 401871, 601771, 636986, 708023, 146104, 441925, 483721, 456410, 315904, 25115, 600914, 285318, 188416, 339901]


In [45]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [47]:
with tf.device('/cpu:0'):
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })

    m.fit()
    print(m.get_score("train"), m.get_score("test"))

self.embed_users['train'].shape =  (249, 100)
self.embed_games.shape =  (785843, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 785843)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (249, 100)


06/08/2024 18:50:39 - WARNING - tensorflow - 5 out of the last 5 calls to <function _BaseOptimizer._update_step_xla at 0x7fcdfe34d040> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details. 


06/08/2024 18:50:39 - WARNING - tensorflow - 6 out of the last 6 calls to <function _BaseOptimizer._update_step_xla at 0x7fcdfe34d040> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details. 



=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0 tf.Tensor(43.044933, shape=(), dtype=float32) 100
slice = key, score = 0.0
np.mean(results), mse, len(results) =  0.00012048192771084337 tf.Tensor(41.940266, shape=(), dtype=float32) 249
slice = train, score = 0.00012048192771084337
np.mean(results), mse, len(results) =  0.00013333333333333334 tf.Tensor(41.75646, shape=(), dtype=float32) 150
slice = test, score = 0.00013333333333333334

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.42029999999999995 tf.Tensor(45.293045, shape=(), dtype=float32) 100
slice = key, score = 0.42029999999999995
np.mean(results), mse, len(results) =  0.5856224899598393 tf.Tensor(43.83515, shape=(), dtype=float32) 249
slice = train, score = 0.5856224899598393
np.mean(results), mse, len(results) =  0.2775333333333333 tf.Tensor(43.328503, shape=(), dtype=float32) 150
slice = test, score = 0.2775333333333333

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.4855000000

In [48]:
del R
del ctx

In [49]:
R = qs_emb[-1000:, :] @ ps_emb.T
R.shape

(1000, 785843)

In [51]:
ctx = EvalContextRelevs(
    R,
    det_attempts=0
)

Best det =  0.0
Current de =  0.0
100 599 300


In [52]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

<ipython-input-17-a13f9b2faac9>:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6717362270450752 0.0008211232 599
np.mean(results), mse, len(results) =  0.5486666666666667 0.0012562973 300


(0.6717362270450752, 0.5486666666666667)

In [53]:
print(ctx.key_games)

[23117, 648111, 352106, 518904, 29215, 785209, 562381, 671373, 75095, 6971, 177361, 254495, 115789, 320357, 185962, 6441, 410817, 169190, 572992, 539555, 397403, 361811, 623852, 178509, 783642, 297896, 671671, 41932, 640208, 146702, 282352, 582947, 368078, 358839, 215277, 140932, 7338, 748977, 263381, 111104, 488496, 122518, 91342, 116301, 701029, 72230, 698820, 246910, 69545, 53879, 335293, 61741, 177942, 724564, 716659, 230399, 76433, 32599, 138260, 67113, 388157, 66081, 242434, 764834, 518530, 516479, 377924, 56800, 660219, 314141, 3624, 709878, 433296, 677008, 781054, 657725, 24964, 456526, 413513, 728527, 370366, 599253, 38208, 354121, 500481, 239210, 597292, 649927, 372657, 86180, 310308, 75868, 601742, 306985, 395653, 425593, 51558, 237868, 435031, 591455]


In [54]:
with tf.device('/cpu:0'):
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })

    m.fit()
    print(m.get_score("train"), m.get_score("test"))

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (785843, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 785843)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0 tf.Tensor(38.330566, shape=(), dtype=float32) 100
slice = key, score = 0.0
np.mean(results), mse, len(results) =  0.0004340567612687813 tf.Tensor(38.79721, shape=(), dtype=float32) 599
slice = train, score = 0.0004340567612687813
np.mean(results), mse, len(results) =  6.666666666666667e-05 tf.Tensor(38.188717, shape=(), dtype=float32) 300
slice = test, score = 6.666666666666667e-05

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.4406999999999999 tf.Tensor(45.58573, shape=(), dtype=float32) 100
slice = key, score = 0.4406999999999999
np.mean(results), mse, len(results) =  0.5519866444073456 tf.Tensor(44.20211, shape=(), dtype=float32) 599
slice = train, sc

In [ ]:
del R
del ctx

In [19]:
test_size = 5000
R = qs_emb[-test_size:, :] @ ps_emb.T
R.shape

(5000, 785843)

In [20]:
ctx = EvalContextRelevs(
    R,
    det_attempts=0
)

Best det =  0.0
Current de =  0.0
100 3399 1500


In [21]:
ctx.key_games = [23117, 299295, 352111, 213468, 639140, 785209, 409314, 286666, 494778, 75095, 572992, 451557, 185962, 38112, 297899, 61928, 641776, 205183, 477125, 466317, 377524, 201402, 42391, 433182, 199386, 446164, 6920, 594232, 587537, 587015, 536583, 637677, 759314, 540897, 532948, 463499, 159522, 128220, 244384, 163594, 24729, 101859, 600407, 107233, 708979, 328323, 573427, 118566, 139, 473313, 716203, 490556, 589528, 534412, 569851, 314457, 641167, 78348, 517003, 320662, 533543, 351357, 214932, 353068, 61849, 612436, 2401, 192669, 486048, 308085, 9290, 674963, 123364, 86308, 412976, 260234, 674649, 572734, 725998, 231231, 778062, 185879, 585311, 352207, 256442, 305389, 722073, 765753, 383635, 219278, 498600, 672959, 184079, 685575, 332614, 566294, 159684, 660854, 162126, 67939]
print(ctx.key_games)

[23117, 299295, 352111, 213468, 639140, 785209, 409314, 286666, 494778, 75095, 572992, 451557, 185962, 38112, 297899, 61928, 641776, 205183, 477125, 466317, 377524, 201402, 42391, 433182, 199386, 446164, 6920, 594232, 587537, 587015, 536583, 637677, 759314, 540897, 532948, 463499, 159522, 128220, 244384, 163594, 24729, 101859, 600407, 107233, 708979, 328323, 573427, 118566, 139, 473313, 716203, 490556, 589528, 534412, 569851, 314457, 641167, 78348, 517003, 320662, 533543, 351357, 214932, 353068, 61849, 612436, 2401, 192669, 486048, 308085, 9290, 674963, 123364, 86308, 412976, 260234, 674649, 572734, 725998, 231231, 778062, 185879, 585311, 352207, 256442, 305389, 722073, 765753, 383635, 219278, 498600, 672959, 184079, 685575, 332614, 566294, 159684, 660854, 162126, 67939]


In [ ]:
ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

In [ ]:
with tf.device('/cpu:0'):
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 1000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })

    m.fit()
    print(m.get_score("train"), m.get_score("test"))

In [55]:
del R
del ctx

In [14]:
test_size = 9650
R = qs_emb[-test_size:, :] @ ps_emb.T
R.shape

(9650, 785843)

In [15]:
ctx = EvalContextRelevs(
    R,
    det_attempts=0
)

Best det =  0.0
Current de =  0.0
100 6654 2895


In [58]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

<ipython-input-17-a13f9b2faac9>:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.5821355575593627 0.0010048697 6654
np.mean(results), mse, len(results) =  0.570020725388601 0.0010468141 2895


(0.5821355575593627, 0.570020725388601)

In [16]:
ctx.key_games = [23117, 299295, 352111, 213468, 639140, 785209, 409314, 286666, 494778, 75095, 572992, 451557, 185962, 38112, 297899, 61928, 641776, 205183, 477125, 466317, 377524, 201402, 42391, 433182, 199386, 446164, 6920, 594232, 587537, 587015, 536583, 637677, 759314, 540897, 532948, 463499, 159522, 128220, 244384, 163594, 24729, 101859, 600407, 107233, 708979, 328323, 573427, 118566, 139, 473313, 716203, 490556, 589528, 534412, 569851, 314457, 641167, 78348, 517003, 320662, 533543, 351357, 214932, 353068, 61849, 612436, 2401, 192669, 486048, 308085, 9290, 674963, 123364, 86308, 412976, 260234, 674649, 572734, 725998, 231231, 778062, 185879, 585311, 352207, 256442, 305389, 722073, 765753, 383635, 219278, 498600, 672959, 184079, 685575, 332614, 566294, 159684, 660854, 162126, 67939]
print(ctx.key_games)

[23117, 299295, 352111, 213468, 639140, 785209, 409314, 286666, 494778, 75095, 572992, 451557, 185962, 38112, 297899, 61928, 641776, 205183, 477125, 466317, 377524, 201402, 42391, 433182, 199386, 446164, 6920, 594232, 587537, 587015, 536583, 637677, 759314, 540897, 532948, 463499, 159522, 128220, 244384, 163594, 24729, 101859, 600407, 107233, 708979, 328323, 573427, 118566, 139, 473313, 716203, 490556, 589528, 534412, 569851, 314457, 641167, 78348, 517003, 320662, 533543, 351357, 214932, 353068, 61849, 612436, 2401, 192669, 486048, 308085, 9290, 674963, 123364, 86308, 412976, 260234, 674649, 572734, 725998, 231231, 778062, 185879, 585311, 352207, 256442, 305389, 722073, 765753, 383635, 219278, 498600, 672959, 184079, 685575, 332614, 566294, 159684, 660854, 162126, 67939]


In [17]:
with tf.device('/cpu:0'):
    N = 100000
    m = CBKnnV0(ctx, fit_kwargs={
        'c': 0,
        'train_matrix': True,
        'train_dssm': True,
        'train_vbias': True,
        'train_popbias': True, 'train_bias': True,
        'verbose': False, 'loss': 'softmax_signed',
        'loss_batch': 128, 'loss_q': 0.99,
        # 'dssm_l2': 5e-5,
        'activation': 'elu',
        'n': N,
        # 'ubatch': 1500,
        'score_verbose': 10000,
        'trainable_items': False,
        "TEinit": "anncur",
        "use_keys_in_train": True, # <<< DIFF HERE
        "learning_rate": 0.0005
    })

    m.fit()
    print(m.get_score("train"), m.get_score("test"))

self.embed_users['train'].shape =  (6654, 100)
self.embed_games.shape =  (785843, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 785843)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (6654, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0 tf.Tensor(39.652737, shape=(), dtype=float32) 100
slice = key, score = 0.0


2024-08-07 06:14:44.139589: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 20915997288 exceeds 10% of free system memory.
2024-08-07 06:14:45.844440: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 20915997288 exceeds 10% of free system memory.
2024-08-07 06:14:54.321492: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 20915997288 exceeds 10% of free system memory.
2024-08-07 06:14:56.022326: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 20915997288 exceeds 10% of free system memory.
2024-08-07 06:15:04.714435: W tensorflow/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 20915997288 exceeds 10% of free system memory.


np.mean(results), mse, len(results) =  0.00017883979561166214 tf.Tensor(40.619205, shape=(), dtype=float32) 6654
slice = train, score = 0.00017883979561166214
np.mean(results), mse, len(results) =  0.0001312607944732297 tf.Tensor(40.550022, shape=(), dtype=float32) 2895
slice = test, score = 0.0001312607944732297

=== Iteration 10000 ===
np.mean(results), mse, len(results) =  0.596 tf.Tensor(285.94806, shape=(), dtype=float32) 100
slice = key, score = 0.596
np.mean(results), mse, len(results) =  0.597400060114217 tf.Tensor(293.215, shape=(), dtype=float32) 6654
slice = train, score = 0.597400060114217
np.mean(results), mse, len(results) =  0.5818445595854922 tf.Tensor(292.47693, shape=(), dtype=float32) 2895
slice = test, score = 0.5818445595854922

=== Iteration 20000 ===
np.mean(results), mse, len(results) =  0.599 tf.Tensor(459.55225, shape=(), dtype=float32) 100
slice = key, score = 0.599
np.mean(results), mse, len(results) =  0.6045491433724076 tf.Tensor(476.4036, shape=(), dtype=

In [42]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (249, 100)
self.embed_games.shape =  (785843, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 785843)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (249, 100)


FailedPreconditionError: DNN library initialization failed. Look at the errors above for more details. [Op:__inference__update_step_xla_953]

# shuffle

In [34]:
R = np.load('./R1000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=True
)

Best det =  0.0
Current de =  0.0
100 599 300


In [35]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.737295492487479 0.0008616203 599
np.mean(results), mse, len(results) =  0.6681666666666666 0.0013216065 300


(0.737295492487479, 0.6681666666666666)

In [36]:
R = np.load('./R1000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 599 300


In [37]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.7360267111853088 0.00086447876 599
np.mean(results), mse, len(results) =  0.6599666666666667 0.0012965548 300


(0.7360267111853088, 0.6599666666666667)

In [38]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (599, 100)
self.embed_games.shape =  (8223, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 8223)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (599, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.019900000000000004 tf.Tensor(41.79086, shape=(), dtype=float32) 100
slice = key, score = 0.019900000000000004
np.mean(results), mse, len(results) =  0.018914858096828045 tf.Tensor(42.32535, shape=(), dtype=float32) 599
slice = train, score = 0.018914858096828045
np.mean(results), mse, len(results) =  0.015333333333333336 tf.Tensor(41.833496, shape=(), dtype=float32) 300
slice = test, score = 0.015333333333333336

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6730999999999999 tf.Tensor(56.058117, shape=(), dtype=float32) 100
slice = key, score = 0.6730999999999999
np.mean(results), mse, len(results) =  0.6961602671118531 tf.Tensor(53.612713, shape=(), dtype=flo


KeyboardInterrupt



In [39]:
R = np.load('./R2000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1299 600


In [40]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/manifold/_spectral_embedding.py:273: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(





model =  <__main__.Popular object at 0x7fb3b6e2ad50>
np.mean(results), mse, len(results) =  0.0131562740569669 0.0052481424 1299
np.mean(results), mse, len(results) =  0.0123 0.0052285064 600
0.0131562740569669 0.0123



model =  AnnCur(140409844430416)
np.mean(results), mse, len(results) =  0.6497613548883756 0.001259694 1299
np.mean(results), mse, len(results) =  0.61785 0.0014691618 600
0.6497613548883756 0.61785



model =  AnnCur(key_games=np.arange(100) - 140409845024464)
np.mean(results), mse, len(results) =  0.6214934565050038 0.0014953915 1299
np.mean(results), mse, len(results) =  0.5832333333333334 0.001780635 600
0.6214934565050038 0.5832333333333334



model =  AnnCur(random - 140409844474704)
np.mean(results), mse, len(results) =  0.6439799846035411 0.0012796524 1299
np.mean(results), mse, len(results) =  0.6204666666666667 0.0014895962 600
0.6439799846035411 0.6204666666666667



model =  AnnCur(K_KMeans - 140409844810960)
np.mean(results), mse, len(results) =  0.6535

In [41]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.687359507313318 0.00095816434 1299
np.mean(results), mse, len(results) =  0.6530333333333335 0.001170445 600


(0.687359507313318, 0.6530333333333335)

In [42]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3566 tf.Tensor(0.006995185094597709, shape=(), dtype=float64) 100
slice = key, score = 0.3566
np.mean(results), mse, len(results) =  0.37841416474210937 tf.Tensor(0.006715046559502405, shape=(), dtype=float64) 1299
slice = train, score = 0.37841416474210937
np.mean(results), mse, len(results) =  0.3667166666666667 tf.Tensor(0.006721360176700356, shape=(), dtype=float64) 600
slice = test, score = 0.3667166666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6606000000000002 tf.Tensor(0.0011121411983254894, shape=(), dtype=float64) 100
slice = key, score = 0.6606000000000002
np.mean(results), mse, len(results) =  0.6864357197844495 tf.Tensor(0.0009623541

(0.7155658198614319, 0.6544833333333333)

In [43]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": True, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3342000000000001 tf.Tensor(0.00812660836522136, shape=(), dtype=float64) 100
slice = key, score = 0.3342000000000001
np.mean(results), mse, len(results) =  0.3460123171670516 tf.Tensor(0.00813912568104136, shape=(), dtype=float64) 1299
slice = train, score = 0.3460123171670516
np.mean(results), mse, len(results) =  0.33418333333333333 tf.Tensor(0.008060184601284569, shape=(), dtype=float64) 600
slice = test, score = 0.33418333333333333

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2625 tf.Tensor(60.13571688826598, shape=(), dtype=float64) 100
slice = key, score = 0.2625
np.mean(results), mse, len(results) =  0.27375673595073136 tf.Tensor(60.16359005293123

(0.6661200923787529, 0.5848)

In [44]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3342 tf.Tensor(0.008297997500201588, shape=(), dtype=float64) 100
slice = key, score = 0.3342
np.mean(results), mse, len(results) =  0.35704387990762126 tf.Tensor(0.007691356497170854, shape=(), dtype=float64) 1299
slice = train, score = 0.35704387990762126
np.mean(results), mse, len(results) =  0.34868333333333335 tf.Tensor(0.007626336463157864, shape=(), dtype=float64) 600
slice = test, score = 0.34868333333333335

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2422 tf.Tensor(57.00378808136002, shape=(), dtype=float64) 100
slice = key, score = 0.2422
np.mean(results), mse, len(results) =  0.26941493456505006 tf.Tensor(55.56765585783994, shape=(), dtype=fl

(0.6716474210931486, 0.5864166666666667)

In [45]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': False,
    'train_popbias': False, 'train_bias': False,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.32959999999999995 tf.Tensor(0.008082917797811014, shape=(), dtype=float64) 100
slice = key, score = 0.32959999999999995
np.mean(results), mse, len(results) =  0.34551963048498846 tf.Tensor(0.0077616020443872, shape=(), dtype=float64) 1299
slice = train, score = 0.34551963048498846
np.mean(results), mse, len(results) =  0.33845 tf.Tensor(0.007700447712871076, shape=(), dtype=float64) 600
slice = test, score = 0.33845

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.6574 tf.Tensor(0.0011422899621397783, shape=(), dtype=float64) 100
slice = key, score = 0.6574
np.mean(results), mse, len(results) =  0.6871747498075443 tf.Tensor(0.000962104919103781, shape=(), dt

(0.7161893764434181, 0.6513666666666666)

In [46]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'mse',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.3051 tf.Tensor(0.009262073986447673, shape=(), dtype=float64) 100
slice = key, score = 0.3051
np.mean(results), mse, len(results) =  0.34113163972286376 tf.Tensor(0.008592258682274378, shape=(), dtype=float64) 1299
slice = train, score = 0.34113163972286376
np.mean(results), mse, len(results) =  0.33048333333333335 tf.Tensor(0.008564824782153313, shape=(), dtype=float64) 600
slice = test, score = 0.33048333333333335

=== Iteration 2000 ===
np.mean(results), mse, len(results) =  0.6576000000000002 tf.Tensor(0.0011449236396029678, shape=(), dtype=float64) 100
slice = key, score = 0.6576000000000002
np.mean(results), mse, len(results) =  0.6862586605080832 tf.Tensor(0.00096465

(0.714203233256351, 0.6515000000000001)

In [47]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': False, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "Winit": "eye",
    "Wfreeze": True,
    "normalize_embs": False,
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (1299, 100)
self.embed_games.shape =  (16433, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 16433)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (1299, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.34409999999999996 tf.Tensor(0.008172939178262067, shape=(), dtype=float64) 100
slice = key, score = 0.34409999999999996
np.mean(results), mse, len(results) =  0.35885296381832177 tf.Tensor(0.00792730245621984, shape=(), dtype=float64) 1299
slice = train, score = 0.35885296381832177
np.mean(results), mse, len(results) =  0.34626666666666667 tf.Tensor(0.0078080144604552575, shape=(), dtype=float64) 600
slice = test, score = 0.34626666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.2305 tf.Tensor(48.52910324848045, shape=(), dtype=float64) 100
slice = key, score = 0.2305
np.mean(results), mse, len(results) =  0.261824480369515 tf.Tensor(50.3829507811

KeyboardInterrupt: 

In [15]:
R = np.load('./R3000train.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [16]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7f40f5c35ed0>
np.mean(results), mse, len(results) =  0.00812406203101551 0.005156699 1999
np.mean(results), mse, len(results) =  0.009422222222222222 0.0051572192 900
0.00812406203101551 0.009422222222222222



model =  AnnCur(139917042483792)
np.mean(results), mse, len(results) =  0.633711855927964 0.001257913 1999
np.mean(results), mse, len(results) =  0.6069222222222223 0.0013856486 900
0.633711855927964 0.6069222222222223



model =  AnnCur(key_games=np.arange(100) - 139916978251152)
np.mean(results), mse, len(results) =  0.59927963981991 0.0015316498 1999
np.mean(results), mse, len(results) =  0.5688777777777778 0.0016877775 900
0.59927963981991 0.5688777777777778



model =  AnnCur(random - 139916978078864)
np.mean(results), mse, len(results) =  0.6311455727863933 0.0012744617 1999
np.mean(results), mse, len(results) =  0.6039888888888888 0.0014088538 900
0.6311455727863933 0.6039888888888888



model =  AnnCur(K_KMeans - 139919554740176)

In [17]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_750529/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6648624312156077 0.000989191 1999
np.mean(results), mse, len(results) =  0.6390555555555555 0.0011131779 900


(0.6648624312156077, 0.6390555555555555)

In [19]:
ctx.set_kmeans_games_as_key(all_from_labels=False)

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET
np.mean(results), mse, len(results) =  0.6281640820410204 0.0012411552 1999
np.mean(results), mse, len(results) =  0.6028111111111111 0.001370198 900


(0.6281640820410204, 0.6028111111111111)

In [90]:
R = np.load('./R3000test.npz')
R_ = R["R"].astype("float64") 
ctx = EvalContextRelevs(
    (R_ - R_.mean()) / R_.std(),
    det_attempts=0,
    shuffle=False
)

Best det =  5.0836383112723515e+59
Current de =  5.0836383112723515e+59
100 1999 900


In [91]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7fb1579a5650>
np.mean(results), mse, len(results) =  0.009444722361180592 0.9567697654965333 1999
np.mean(results), mse, len(results) =  0.009166666666666667 0.971850685741731 900
0.009444722361180592 0.009166666666666667



model =  AnnCur(140394774089168)
np.mean(results), mse, len(results) =  0.6309604802401201 0.2417310715297432 1999
np.mean(results), mse, len(results) =  0.6147777777777778 0.26450940184444943 900
0.6309604802401201 0.6147777777777778



model =  AnnCur(key_games=np.arange(100) - 140409843271824)
np.mean(results), mse, len(results) =  0.5956728364182091 0.2882643239306443 1999
np.mean(results), mse, len(results) =  0.5721444444444443 0.31852632814725745 900
0.5956728364182091 0.5721444444444443



model =  AnnCur(random - 140406345444496)
np.mean(results), mse, len(results) =  0.633896948474237 0.23576882801676524 1999
np.mean(results), mse, len(results) =  0.613588888888889 0.2624383474778105 900
0.633896948474237 0.613588

In [ ]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

In [50]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6689094547273637 0.000980972 1999
np.mean(results), mse, len(results) =  0.6458777777777778 0.0011021171 900


(0.6689094547273637, 0.6458777777777778)

In [53]:
ctx.set_kmeans_games_as_key()

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET
np.mean(results), mse, len(results) =  0.6234267133566783 0.001265143 1999
np.mean(results), mse, len(results) =  0.6084555555555554 0.0013879173 900


(0.6234267133566783, 0.6084555555555554)

In [52]:
dir(ctx)

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'games_count',
 'get_kmeans_games',
 'get_relevs',
 'get_requests',
 'get_top_games',
 'key_games',
 'key_relevs',
 'key_size',
 'relevs',
 'reqs_count',
 'set_kmeans_games_as_key',
 'set_top_games_as_key',
 'slices',
 'test_relevs',
 'train_relevs',
 'train_split',
 'try_det_attempts']

In [54]:
R = np.load('./R3000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=True
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [55]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_425468/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6721460730365183 0.0009835806 1999
np.mean(results), mse, len(results) =  0.6366222222222223 0.0011193526 900


(0.6721460730365183, 0.6366222222222223)

In [56]:
ctx.set_kmeans_games_as_key()

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KEY_SET
np.mean(results), mse, len(results) =  0.6298099049524764 0.0012462635 1999
np.mean(results), mse, len(results) =  0.5925555555555555 0.0013914837 900


(0.6298099049524764, 0.5925555555555555)

In [87]:
kr = ctx.relevs[:100, ctx.key_games].astype("float64")

In [85]:
kr = (kr - kr.mean()) 

In [88]:
np.linalg.det(kr)

8.738492240116713e-53

In [83]:
kr.dtype

dtype('float32')

In [20]:
R = np.load('./R3000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 1999 900


In [21]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7f40e95d2e90>
np.mean(results), mse, len(results) =  0.009444722361180592 0.005165653 1999
np.mean(results), mse, len(results) =  0.009166666666666667 0.0052470677 900
0.009444722361180592 0.009166666666666667



model =  AnnCur(139916770041104)
np.mean(results), mse, len(results) =  0.6352426213106552 0.0012897311 1999
np.mean(results), mse, len(results) =  0.6164111111111111 0.0014229873 900
0.6352426213106552 0.6164111111111111



model =  AnnCur(key_games=np.arange(100) - 139916769195984)
np.mean(results), mse, len(results) =  0.5953476738369184 0.0015576119 1999
np.mean(results), mse, len(results) =  0.5720444444444445 0.0017203941 900
0.5953476738369184 0.5720444444444445



model =  AnnCur(random - 139916769190096)
np.mean(results), mse, len(results) =  0.633671835917959 0.0012710841 1999
np.mean(results), mse, len(results) =  0.6130444444444445 0.0014154438 900
0.633671835917959 0.6130444444444445



model =  AnnCur(K_KMeans - 139916770

In [22]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_750529/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6689094547273637 0.000980972 1999
np.mean(results), mse, len(results) =  0.6458777777777778 0.0011021171 900


(0.6689094547273637, 0.6458777777777778)

## R5000test

In [15]:
R = np.load('./R5000test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 3399 1500


In [24]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7f40f5c4af90>
np.mean(results), mse, len(results) =  0.005445719329214474 0.005149558 3399
np.mean(results), mse, len(results) =  0.005253333333333334 0.0051724273 1500
0.005445719329214474 0.005253333333333334



model =  AnnCur(139916976503504)
np.mean(results), mse, len(results) =  0.6149132097675788 0.0013162897 3399
np.mean(results), mse, len(results) =  0.5955066666666666 0.0013957007 1500
0.6149132097675788 0.5955066666666666



model =  AnnCur(key_games=np.arange(100) - 139916976500752)
np.mean(results), mse, len(results) =  0.5783995292733157 0.0015776153 3399
np.mean(results), mse, len(results) =  0.5589733333333333 0.0016660148 1500
0.5783995292733157 0.5589733333333333



model =  AnnCur(random - 139916770160016)
np.mean(results), mse, len(results) =  0.6142071197411003 0.0012941539 3399
np.mean(results), mse, len(results) =  0.59584 0.0013727732 1500
0.6142071197411003 0.59584



model =  AnnCur(K_KMeans - 139916770086288)
np.mean(

In [18]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_602384/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6448455428067078 0.0010061534 3399
np.mean(results), mse, len(results) =  0.6315666666666667 0.0010672992 1500


(0.6448455428067078, 0.6315666666666667)

In [19]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (3399, 100)
self.embed_games.shape =  (40718, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 40718)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (3399, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.004699999999999999 tf.Tensor(39.029186, shape=(), dtype=float32) 100
slice = key, score = 0.004699999999999999
np.mean(results), mse, len(results) =  0.002362459546925567 tf.Tensor(37.687843, shape=(), dtype=float32) 3399
slice = train, score = 0.002362459546925567
np.mean(results), mse, len(results) =  0.002426666666666667 tf.Tensor(37.627937, shape=(), dtype=float32) 1500
slice = test, score = 0.002426666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5627 tf.Tensor(48.584892, shape=(), dtype=float32) 100
slice = key, score = 0.5627
np.mean(results), mse, len(results) =  0.6158399529273316 tf.Tensor(47.981865, shape=(), dtype=float32) 3399
slice

(0.7013268608414239, 0.6552266666666667)

Key 0.6306,

Train 0.7015269196822596,

**Test 0.65594**

In [21]:
# s = """ """

In [21]:
def get_train_test(s):
    v = s.strip().split("\n")
    try:
        key = [v_i for v_i in v if v_i.startswith("slice = key")][0]
        train = [v_i for v_i in v if v_i.startswith("slice = train")][0]
        test = [v_i for v_i in v if v_i.startswith("slice = test")][0]
        return float(key.split(" = ")[-1]), float(train.split(" = ")[-1]), float(test.split(" = ")[-1])
    except Exception as e:
        print("wtf")
        return (-1, -1)
    
def get_best(s):
    return sorted([get_train_test(x) for x in s.strip().split("=== Iteration") if x], reverse=True)[:3]

In [23]:
get_best(s)

[(0.6306, 0.7015269196822596, 0.65594),
 (0.6299999999999999, 0.7015328037658135, 0.6571733333333333),
 (0.6295000000000001, 0.6970108855545748, 0.6534133333333333)]

In [16]:
ctx.set_kmeans_games_as_key(all_from_labels=True)

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.6009002647837599 0.001285355 3399
np.mean(results), mse, len(results) =  0.58148 0.0013592697 1500


(0.6009002647837599, 0.58148)

In [17]:
ctx.set_kmeans_games_as_key(all_from_labels=False)

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


np.mean(results), mse, len(results) =  0.6027596351868196 0.0012819113 3399
np.mean(results), mse, len(results) =  0.5840933333333334 0.0013497035 1500


(0.6027596351868196, 0.5840933333333334)

In [18]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (3399, 100)
self.embed_games.shape =  (40718, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 40718)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (3399, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0034000000000000002 tf.Tensor(44.46507, shape=(), dtype=float32) 100
slice = key, score = 0.0034000000000000002
np.mean(results), mse, len(results) =  0.0036893203883495147 tf.Tensor(42.645073, shape=(), dtype=float32) 3399
slice = train, score = 0.0036893203883495147
np.mean(results), mse, len(results) =  0.0033866666666666667 tf.Tensor(42.09904, shape=(), dtype=float32) 1500
slice = test, score = 0.0033866666666666667

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5429 tf.Tensor(51.350933, shape=(), dtype=float32) 100
slice = key, score = 0.5429
np.mean(results), mse, len(results) =  0.5841100323624595 tf.Tensor(52.668064, shape=(), dtype=float32) 3399
s

(0.6748837893498089, 0.6147)

[(0.5937, 0.6748337746395998, **0.6141733333333333**),
 (0.5917000000000001, 0.6682082965578112, 0.61236),
 (0.5916, 0.6698205354516035, 0.61252)]

In [19]:
# s = """"""

In [22]:
get_best(s)

[(0.5937, 0.6748337746395998, 0.6141733333333333),
 (0.5917000000000001, 0.6682082965578112, 0.61236),
 (0.5916, 0.6698205354516035, 0.61252)]

## R9650test

In [26]:
R = np.load('./R9650test.npz')
ctx = EvalContextRelevs(
    R["R"],
    det_attempts=0,
    shuffle=False
)

Best det =  0.0
Current de =  0.0
100 6654 2895


In [27]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
/home/shevkunov/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:1930: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=3)





model =  <__main__.Popular object at 0x7fa73ffb53d0>
np.mean(results), mse, len(results) =  0.0026615569582206194 0.0051135845 6654
np.mean(results), mse, len(results) =  0.0021796200345423145 0.005103332 2895
0.0026615569582206194 0.0021796200345423145



model =  AnnCur(140356329099280)
np.mean(results), mse, len(results) =  0.5931364592726179 0.0013309367 6654
np.mean(results), mse, len(results) =  0.582846286701209 0.0013727237 2895
0.5931364592726179 0.582846286701209



model =  AnnCur(key_games=np.arange(100) - 140356483882640)
np.mean(results), mse, len(results) =  0.5607048391944696 0.0015862373 6654
np.mean(results), mse, len(results) =  0.5490604490500863 0.0016326709 2895
0.5607048391944696 0.5490604490500863



model =  AnnCur(random - 140356329589072)
np.mean(results), mse, len(results) =  0.5874256086564472 0.0013426415 6654
np.mean(results), mse, len(results) =  0.5773436960276339 0.0013754577 2895
0.5874256086564472 0.5773436960276339



model =  AnnCur(K_KMeans - 1

In [28]:
t = ctx.get_relevs("train").T
ctx.key_games = list(coitem_algorithm(100, t, t, 1e-8, eps=1e9)[0])

print("KEY_SET")

ma = AnnCUR(ctx, key_games=ctx.key_games)

ma.fit()
ma.get_score("train"), ma.get_score("test")

/tmp/ipykernel_668938/4274979955.py:15: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for t in tqdm.tqdm_notebook(range(n_support)):


  0%|          | 0/100 [00:00<?, ?it/s]

KEY_SET
np.mean(results), mse, len(results) =  0.6206026450255485 0.0010117242 6654
np.mean(results), mse, len(results) =  0.6100034542314335 0.0010527462 2895


(0.6206026450255485, 0.6100034542314335)

In [29]:
N = 100000
m = CBKnnV0(ctx, fit_kwargs={
    'c': 0,
    'train_matrix': True,
    'train_dssm': True,
    'train_vbias': True,
    'train_popbias': True, 'train_bias': True,
    'verbose': False, 'loss': 'softmax_signed',
    'loss_batch': 128, 'loss_q': 0.99,
    # 'dssm_l2': 5e-5,
    'activation': 'elu',
    'n': N,
    # 'ubatch': 1500,
    'score_verbose': 1000,
    'trainable_items': False,
    "TEinit": "anncur",
    "use_keys_in_train": False, # <<< DIFF HERE
    "learning_rate": 0.0005
})

m.fit()
m.get_score("train"), m.get_score("test")

self.embed_users['train'].shape =  (6654, 100)
self.embed_games.shape =  (77897, 100)
ANNCur init
self.games2users.shape =  (100, 100)
self.core_users_scores.shape =  (100, 77897)
self.core_users_embs.shape =  (100, 100)
self.ge_users.shape =  (6654, 100)

=== Iteration 0 ===
np.mean(results), mse, len(results) =  0.0025 tf.Tensor(39.800236, shape=(), dtype=float32) 100
slice = key, score = 0.0025
np.mean(results), mse, len(results) =  0.0023309287646528405 tf.Tensor(39.002846, shape=(), dtype=float32) 6654
slice = train, score = 0.0023309287646528405
np.mean(results), mse, len(results) =  0.001830742659758204 tf.Tensor(38.9024, shape=(), dtype=float32) 2895
slice = test, score = 0.001830742659758204

=== Iteration 1000 ===
np.mean(results), mse, len(results) =  0.5559000000000001 tf.Tensor(51.01326, shape=(), dtype=float32) 100
slice = key, score = 0.5559000000000001
np.mean(results), mse, len(results) =  0.5929486023444545 tf.Tensor(50.45188, shape=(), dtype=float32) 6654
slice = tra

(0.6735287045386233, 0.6483868739205527)

In [33]:
p = Popular(ctx)
p.fit()

In [40]:
ma = AnnCUR(ctx, key_games=p.top_logits.argsort()[:100])

ma.fit()
ma.get_score("train"), ma.get_score("test")

np.mean(results), mse, len(results) =  0.5661602043883378 0.0017261917 6654
np.mean(results), mse, len(results) =  0.553644214162349 0.0017789402 2895


(0.5661602043883378, 0.553644214162349)

In [42]:
np.random.seed(17)

l = list()
for _ in range(10):
    ma = AnnCUR(ctx, key_games=np.random.choice(np.arange(ctx.games_count), 100, replace=False))

    ma.fit()
    tr, te = ma.get_score("train"), ma.get_score("test")
    print(tr, te)
    l.append(te)
    
np.mean(l)

np.mean(results), mse, len(results) =  0.5874256086564472 0.0013426415 6654
np.mean(results), mse, len(results) =  0.5773436960276339 0.0013754577 2895
0.5874256086564472 0.5773436960276339
np.mean(results), mse, len(results) =  0.5904508566275923 0.0013353806 6654
np.mean(results), mse, len(results) =  0.5779758203799654 0.0013791524 2895
0.5904508566275923 0.5779758203799654
np.mean(results), mse, len(results) =  0.596355575593628 0.0013101283 6654
np.mean(results), mse, len(results) =  0.5849775474956822 0.0013571075 2895
0.596355575593628 0.5849775474956822
np.mean(results), mse, len(results) =  0.5875082657048392 0.0013476225 6654
np.mean(results), mse, len(results) =  0.578607944732297 0.0013864961 2895
0.5875082657048392 0.578607944732297
np.mean(results), mse, len(results) =  0.5907108506161708 0.0013493096 6654
np.mean(results), mse, len(results) =  0.5787184801381692 0.0013923912 2895
0.5907108506161708 0.5787184801381692
np.mean(results), mse, len(results) =  0.5940847610459

0.58020103626943